In [ ]:
from __future__ import annotations

from typing import Any


QUIZ_SERVICE_VERSION = "2026-09-16-budget-v2"


class QuizService:
    """
    Coordinates adaptive quiz generation.

    Responsibilities:
    - retrieve project knowledge
    - collect the learner's current mastery state
    - collect recent assessment history
    - pass the combined learning context to QuizGenerator
    """

    def __init__(self, database):
        self.database = database

    def generate_quiz(
        self,
        project_id: str,
        user_id: str,
        concepts: list[str] | None = None,
        question_count: int = 5,
        difficulty: str = "medium",
    ):

        from app.services.ai_service import AIService
        from app.services.retrieval_service import RetrievalService
        from app.ai.quiz_generator import QuizGenerator

        if question_count < 1:
            raise ValueError("question_count must be at least 1.")

        if question_count > 20:
            raise ValueError("question_count cannot exceed 20.")

        allowed_difficulties = {
            "easy",
            "medium",
            "hard",
        }

        if difficulty not in allowed_difficulties:
            raise ValueError(
                f"Invalid difficulty '{difficulty}'. "
                f"Expected one of: {sorted(allowed_difficulties)}."
            )

        concepts = concepts or []

        # ----------------------------------------------------
        # RETRIEVE PROJECT KNOWLEDGE
        # ----------------------------------------------------

        retrieval = RetrievalService(
            database=self.database
        )

        evidence_chunks = retrieval.get_project_chunks(
            project_id=project_id,
            user_id=user_id,
            limit=10,
        )

        # ----------------------------------------------------
        # RETRIEVE LEARNING STATE
        # ----------------------------------------------------

        mastery_documents = list(
            self.database.collection("mastery").find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            ).sort(
                "score",
                1,
            ).limit(10)
        )

        # ----------------------------------------------------
        # RETRIEVE RECENT ASSESSMENT HISTORY
        # ----------------------------------------------------

        assessment_documents = list(
            self.database.collection("assessments").find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            ).sort(
                "created_at",
                -1,
            ).limit(10)
        )

        # ----------------------------------------------------
        # RETRIEVE RECENT ACTIVITY
        # ----------------------------------------------------

        activity_documents = list(
            self.database.collection("activities").find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            ).sort(
                "created_at",
                -1,
            ).limit(10)
        )

        # ----------------------------------------------------
        # AI GENERATION
        # ----------------------------------------------------

        ai = AIService(
            database=self.database
        )

        generator = QuizGenerator(
            ai_service=ai
        )

        return generator.generate(
            evidence_chunks=evidence_chunks,
            concepts=concepts,
            question_count=question_count,
            difficulty=difficulty,
            mastery=mastery_documents,
            recent_performance=assessment_documents,
            recent_activity=activity_documents,
            user_id=user_id,
            project_id=project_id,
        )